<a href="https://colab.research.google.com/github/ZarinRitu32/Myopia_ML/blob/main/SoftCom_06_Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SoftCom myopia-risk benchmark - Section 6 - Export & reproducibility

Replication and extension of Mu et al. 2024 (Front. Med. 11:1482788) on the provided **hybrid cohort (N=1500)**. Pipeline validation only - no clinical claims.

**Pipeline:** SoftCom_01_Data_Validation -> SoftCom_02_Preprocessing -> SoftCom_03_Benchmarking -> SoftCom_04_Explainable_AI -> SoftCom_05_Figures -> **SoftCom_06_Export**

All inputs and outputs live in Google Drive at `MyDrive/SoftCom_Myopia/`. Each notebook checks `artifacts/manifest.json` and stops if an upstream step has not run or its files have changed.

| | |
|---|---|
| **Reads** | outputs of 01-05 |
| **Writes** | `results.json`, `RESULTS_INDEX.md`, `data/processed/cohort_clean_v1.csv` |
| **Next** | none (final notebook) |

Runtime: Colab CPU is enough. Use *Runtime -> Run all*; accept the Drive prompt in the setup cell. For a quick trial run, set `"FAST_MODE": True` in the CONFIG cell of **every** notebook (fewer CV repeats/bootstraps); set it back to `False` for the reported results.

## 0. Install dependencies and set up Google Drive

In [ ]:
%pip install -q "shap>=0.47" xgboost lightgbm statsmodels imbalanced-learn

In [ ]:
"""Shared setup (identical in every SoftCom notebook).

Mounts Google Drive, defines CONFIG, the Drive folder layout, the run manifest
that links the six notebooks together, seeding, logging and figure helpers.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ----------------------------------------------------------------------------- CONFIG
CONFIG: Dict[str, Any] = {
    "SEED": 42,
    "TEST_SIZE": 0.3,
    "N_SPLITS": 5,
    "N_REPEATS": 10,
    "N_BOOTSTRAP": 2000,
    "EXPECTED_N_ROWS": 1500,
    "DATA_FILENAME": "full_myopia_hybrid_dataset_N1500.csv",
    "PILOT_FILENAME": "pilot_real.csv",          # optional 73-row pilot; never merged
    "TARGET": "is_myopic",
    "SOURCE_LABEL": "hybrid",
    "COHORT_LABEL": "Hybrid cohort (N=1500) - pipeline validation only, not clinical evidence",
    "DRIVE_PROJECT_DIR": "SoftCom_Myopia",       # folder inside MyDrive holding everything
    "USE_SMOTE": False,                          # SMOTE inside pipeline only (imblearn)
    "USE_LASSO_SELECTION": True,                 # L1-logistic SelectFromModel inside pipeline
    "ABLATION": True,                            # also run FT-Transformer without attention
    "DEEP_MAX_EPOCHS": 200,
    "DEEP_PATIENCE": 20,
    "DEEP_BATCH_SIZE": 64,
    "PERM_N_REPEATS": 20,
    "SHAP_BACKGROUND": 100,
    "SHAP_N_EXPLAIN": 150,
    "SHAP_KERNEL_NSAMPLES": 200,
    "FAST_MODE": False,                          # True = quick trial run (same value in all notebooks)
}
CONFIG["FAST_MODE"] |= os.environ.get("SOFTCOM_FAST", "0") == "1"  # local smoke tests
if CONFIG["FAST_MODE"]:
    CONFIG.update(N_REPEATS=2, N_BOOTSTRAP=200, DEEP_MAX_EPOCHS=25, DEEP_PATIENCE=5,
                  PERM_N_REPEATS=3, SHAP_BACKGROUND=20, SHAP_N_EXPLAIN=20,
                  SHAP_KERNEL_NSAMPLES=60)
SEED: int = CONFIG["SEED"]

NOTEBOOKS: Dict[str, str] = {
    "01_data": "SoftCom_01_Data_Validation.ipynb",
    "02_preprocessing": "SoftCom_02_Preprocessing.ipynb",
    "03_benchmark": "SoftCom_03_Benchmarking.ipynb",
    "04_xai": "SoftCom_04_Explainable_AI.ipynb",
    "05_figures": "SoftCom_05_Figures.ipynb",
    "06_export": "SoftCom_06_Export.ipynb",
}

# ----------------------------------------------------------------------------- Drive
LAUNCH_DIR: Path = Path.cwd()
try:
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive")
        except Exception as exc:  # pragma: no cover - interactive auth
            raise RuntimeError(
                "Google Drive mount failed. Re-run this cell and accept the Drive "
                f"authorisation prompt. Original error: {exc}"
            ) from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive") / CONFIG["DRIVE_PROJECT_DIR"]
else:
    PROJECT_ROOT = Path(os.environ.get("SOFTCOM_ROOT", LAUNCH_DIR / CONFIG["DRIVE_PROJECT_DIR"])).resolve()

DIRS: Dict[str, Path] = {
    name: PROJECT_ROOT / sub
    for name, sub in {
        "raw": "data/raw",
        "processed": "data/processed",
        "models": "models",
        "predictions": "predictions",
        "tables": "tables",
        "figures": "figures",
        "artifacts": "artifacts",
        "logs": "logs",
    }.items()
}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)  # so ./figures/ and ./tables/ resolve inside Google Drive
MANIFEST_PATH: Path = DIRS["artifacts"] / "manifest.json"


# ----------------------------------------------------------------------------- helpers
def set_global_seed(seed: int = SEED) -> None:
    """Seed python, numpy and (if installed) torch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch  # type: ignore

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass


def log(step: str, message: str) -> None:
    """Print a message and append it to logs/<step>.log on Drive."""
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    print(message)
    with open(DIRS["logs"] / f"{step}.log", "a", encoding="utf-8") as fh:
        fh.write(f"[{stamp}] {message}\n")


def sha256_file(path: Path) -> str:
    """Return the SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def rel(path: Path) -> str:
    """Path relative to the project root (as stored in the manifest)."""
    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def _atomic_write_json(obj: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


def save_json(obj: Any, path: Path) -> Path:
    """Write JSON atomically (numpy/pandas values are stringified if needed)."""
    _atomic_write_json(obj, path)
    return path


def load_manifest() -> Dict[str, Any]:
    """Load the cross-notebook run manifest (empty skeleton if absent)."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, encoding="utf-8") as fh:
            return json.load(fh)
    return {"project": "SoftCom myopia benchmark", "steps": {}}


def register_outputs(step: str, paths: Iterable[Path], extra: Optional[Dict[str, Any]] = None) -> None:
    """Record a finished notebook step and the SHA-256 of every file it wrote."""
    manifest = load_manifest()
    files = {}
    for p in paths:
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"[{step}] expected output was not written: {p}")
        files[rel(p)] = sha256_file(p)
    manifest["steps"][step] = {
        "notebook": NOTEBOOKS[step],
        "status": "ok",
        "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "fast_mode": CONFIG["FAST_MODE"],
        "outputs": files,
        "extra": extra or {},
    }
    _atomic_write_json(manifest, MANIFEST_PATH)
    print(f"Manifest updated: step '{step}' -> {len(files)} files registered.")


def require_upstream(step: str, needed: Sequence[str]) -> None:
    """Fail loudly unless `step` ran successfully and its files are unchanged.

    `needed` are project-relative paths that this notebook will read.
    """
    manifest = load_manifest()
    info = manifest["steps"].get(step)
    if info is None or info.get("status") != "ok":
        raise RuntimeError(
            f"Upstream step '{step}' has not completed. Open and 'Run all' "
            f"{NOTEBOOKS[step]} first (results folder: {PROJECT_ROOT})."
        )
    for r in needed:
        p = PROJECT_ROOT / r
        if not p.exists():
            raise FileNotFoundError(f"Missing upstream file {r}; re-run {NOTEBOOKS[step]}.")
        expected = info["outputs"].get(r)
        if expected is None:
            raise RuntimeError(f"{r} is not registered by {NOTEBOOKS[step]}; re-run it.")
        if sha256_file(p) != expected:
            raise RuntimeError(
                f"{r} changed after {NOTEBOOKS[step]} finished (hash drift). "
                f"Re-run {NOTEBOOKS[step]} and the notebooks after it."
            )
    if info.get("fast_mode") != CONFIG["FAST_MODE"]:
        print(f"WARNING: {step} ran with FAST_MODE={info.get('fast_mode')}, this notebook uses "
              f"FAST_MODE={CONFIG['FAST_MODE']}.")
    print(f"Upstream OK: {step} ({info['finished_at']}), {len(needed)} files verified.")


sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "legend.fontsize": 10, "savefig.bbox": "tight"})


def save_figure(fig: plt.Figure, name: str, footnote: bool = True) -> List[Path]:
    """Save `fig` as 300-DPI PNG and SVG in ./figures/ (on Drive) and close it."""
    if footnote:  # place below everything already drawn (rotated tick labels included)
        bottom = fig.get_tightbbox(fig.canvas.get_renderer()).y0 / fig.get_figheight()
        fig.text(0.995, min(bottom, 0.0) - 0.01, CONFIG["COHORT_LABEL"], ha="right", va="top", fontsize=8, color="0.45")
    out = []
    try:
        for ext in ("png", "svg"):
            p = DIRS["figures"] / f"{name}.{ext}"
            fig.savefig(p, dpi=300, bbox_inches="tight")
            out.append(p)
    except OSError as exc:
        raise RuntimeError(f"Could not write figure {name} to {DIRS['figures']}: {exc}") from exc
    finally:
        plt.close(fig)
    print(f"Saved figure: figures/{name}.png + .svg")
    return out


def show_saved(name: str) -> None:
    """Display a saved PNG inline (Colab/Jupyter only)."""
    try:
        from IPython.display import Image, display  # type: ignore

        if "ipykernel" in sys.modules:
            display(Image(filename=str(DIRS["figures"] / f"{name}.png"), width=720))
    except ImportError:
        pass


set_global_seed(SEED)
print(f"Colab: {IN_COLAB} | results folder: {PROJECT_ROOT} | FAST_MODE={CONFIG['FAST_MODE']}")

Mounted at /content/drive
Colab: True | results folder: /content/drive/MyDrive/SoftCom_Myopia | FAST_MODE=False


## 6.1 Verify every upstream notebook finished and its files are unchanged

In [ ]:
import importlib.metadata as md
import platform

STEP = "06_export"
manifest = load_manifest()
for step in ["01_data", "02_preprocessing", "03_benchmark", "04_xai", "05_figures"]:
    require_upstream(step, list(manifest["steps"].get(step, {}).get("outputs", {})))

Upstream OK: 01_data (2026-09-24T16:48:56+00:00), 5 files verified.
Upstream OK: 02_preprocessing (2026-09-24T16:50:58+00:00), 4 files verified.
Upstream OK: 03_benchmark (2026-09-24T17:19:57+00:00), 28 files verified.
Upstream OK: 04_xai (2026-09-24T17:29:14+00:00), 14 files verified.
Upstream OK: 05_figures (2026-09-24T17:34:02+00:00), 6 files verified.


## 6.2 Assemble results.json

In [ ]:
def read_json(path: Path) -> Any:
    """Load a JSON artifact."""
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def versions(pkgs: Sequence[str]) -> Dict[str, str]:
    """Installed versions of the packages this project depends on (pip-freeze subset)."""
    out = {}
    for p in pkgs:
        try:
            out[p] = md.version(p)
        except md.PackageNotFoundError:
            out[p] = "not installed"
    return out


validation = read_json(DIRS["artifacts"] / "validation_report.json")
fspec = read_json(DIRS["artifacts"] / "feature_spec.json")
bmeta = read_json(DIRS["artifacts"] / "benchmark_meta.json")
xmeta = read_json(DIRS["artifacts"] / "xai_meta.json")
table2 = pd.read_csv(DIRS["tables"] / "table2_model_performance.csv", index_col="Model")
holdout = pd.read_csv(DIRS["tables"] / "table2c_holdout_detail.csv", index_col="Model")
cv_summary = pd.read_csv(DIRS["tables"] / "table2b_repeated_cv_summary.csv", header=[0, 1], index_col=0)
delong = pd.read_csv(DIRS["tables"] / "table3_delong_pairwise.csv")
ors = pd.read_csv(DIRS["tables"] / "table4_odds_ratios.csv")
perm = pd.read_csv(DIRS["tables"] / "table5_permutation_importance.csv")
shap_tab = pd.read_csv(DIRS["tables"] / "table6_shap_mean_abs.csv", index_col=0)
cv_flat = {m: {f"{metric}_{stat}": float(cv_summary.loc[m, (metric, stat)]) for metric, stat in cv_summary.columns}
           for m in cv_summary.index}

results = {
    "label": CONFIG["COHORT_LABEL"],
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": CONFIG,
    "seeds": {"global": SEED, "train_test_split": SEED, "repeated_cv": SEED, "bootstrap": SEED,
              "permutation": SEED, "shap_sampling": SEED, "torch": SEED},
    "dataset": {"file": CONFIG["DATA_FILENAME"], "sha256": validation["raw_sha256"], "n_rows": validation["n_rows"],
                "prevalence": validation["prevalence"], "features": fspec["numeric"] + fspec["categorical"],
                "excluded": fspec["excluded"], "split_hash": fspec["split_hash"]},
    "validation": validation,
    "fold_indices_hash": bmeta["fold_hash"],
    "best_model": bmeta["best_model"],
    "table2_model_performance": table2.reset_index().to_dict(orient="records"),
    "holdout_detail": holdout.reset_index().to_dict(orient="records"),
    "repeated_cv": cv_flat,
    "delong_pairwise": delong.to_dict(orient="records"),
    "odds_ratios": ors.to_dict(orient="records"),
    "permutation_importance": perm.to_dict(orient="records"),
    "shap_mean_abs": shap_tab.reset_index().rename(columns={"index": "feature"}).to_dict(orient="records"),
    "xai": xmeta,
    "compute_cost": {m: {k: v for k, v in d.items() if k in ("tuning_seconds", "cv_seconds", "n_grid_configs", "n_parameters")}
                     for m, d in bmeta["models"].items()},
    "best_params": {m: d["best_params"] for m, d in bmeta["models"].items()},
    "skipped_models": bmeta["skipped"],
    "environment": {"python": sys.version.split()[0], "platform": platform.platform(), "colab": IN_COLAB,
                    "packages": versions(["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "shap",
                                          "statsmodels", "imbalanced-learn", "torch", "matplotlib", "seaborn", "joblib"])},
    "manifest": manifest,
}

## 6.3 Export files and check completeness

In [ ]:
EXPECTED_FIGS = ["fig2_prevalence", "fig3_or_forest", "fig4_distributions", "fig5_roc_train_test",
                 "fig6_importance", "fig7_shap_summary", "fig8_heatmap"]
fig_check = {f: all((DIRS["figures"] / f"{f}.{e}").exists() for e in ("png", "svg")) for f in EXPECTED_FIGS}
results["figure_check"] = fig_check

cohort = pd.read_csv(DIRS["processed"] / "cohort_features.csv")
cohort["source"] = CONFIG["SOURCE_LABEL"]
out = {"results": PROJECT_ROOT / "results.json", "cohort": DIRS["processed"] / "cohort_clean_v1.csv",
       "index": PROJECT_ROOT / "RESULTS_INDEX.md"}
try:
    cohort.to_csv(out["cohort"], index=False)
    save_json(results, out["results"])
except OSError as exc:
    raise RuntimeError(f"Writing final exports to Drive failed: {exc}") from exc

files = sorted(p for p in PROJECT_ROOT.rglob("*") if p.is_file() and p.suffix not in (".tmp",) and p != out["index"])
lines = [f"# SoftCom myopia benchmark - results index", "", f"_{CONFIG['COHORT_LABEL']}_", "",
         f"Generated {results['generated_at']} | best model: **{bmeta['best_model']}**", "", "| file | size (KB) |", "|---|---|"]
lines += [f"| {rel(p)} | {p.stat().st_size / 1024:.1f} |" for p in files]
out["index"].write_text("\n".join(lines) + "\n", encoding="utf-8")
register_outputs(STEP, list(out.values()), {"figures_complete": all(fig_check.values())})
missing_figs = [f for f, ok in fig_check.items() if not ok]
if missing_figs:
    print(f"WARNING: missing figures {missing_figs}")

Manifest updated: step '06_export' -> 3 files registered.


## 6.4 Run summary

In [ ]:
best = bmeta["best_model"]
n_sig = int(delong["significant_0.05"].sum())
summary = [
    f"1. Data: {CONFIG['DATA_FILENAME']} | rows={validation['n_rows']} | prevalence={validation['prevalence']:.3f}",
    f"2. Validation: {validation['status']}",
    f"3. Features: {len(results['dataset']['features'])} used, {len(fspec['excluded'])} excluded (SE excluded: label leakage)",
    f"4. Split: train={fspec['n_train']} test={fspec['n_test']} | CV {CONFIG['N_SPLITS']}x{CONFIG['N_REPEATS']} | fold hash {bmeta['fold_hash'][:12]}",
    f"5. Models ({len(bmeta['models'])}): {', '.join(bmeta['models'])}" + (f" | skipped: {', '.join(bmeta['skipped'])}" if bmeta['skipped'] else ""),
    f"6. Best (repeated-CV AUC): {best} = {cv_flat[best]['roc_auc_mean']:.3f} +- {cv_flat[best]['roc_auc_std']:.3f}",
    f"7. {best} holdout AUC = {holdout.loc[best, 'roc_auc']:.3f} (95% CI {holdout.loc[best, 'auc_ci_low']:.3f}-{holdout.loc[best, 'auc_ci_high']:.3f})",
    f"8. DeLong (Holm): {n_sig}/{len(delong)} model pairs differ at alpha=0.05",
    f"9. Figures complete: {all(fig_check.values())} ({sum(fig_check.values())}/{len(fig_check)}) | SHAP primary: {xmeta['shap_primary']}",
    f"10. Files written to {PROJECT_ROOT} ({len(files) + 1} files) incl. results.json, RESULTS_INDEX.md",
]
print("=" * 70)
print("\n".join(summary))
print("=" * 70)
print(CONFIG["COHORT_LABEL"])

1. Data: full_myopia_hybrid_dataset_N1500.csv | rows=1500 | prevalence=0.312
2. Validation: PASS with warnings
3. Features: 12 used, 6 excluded (SE excluded: label leakage)
4. Split: train=1050 test=450 | CV 5x10 | fold hash bd44cda8406f
5. Models (10): LR, DT, RF, XGBoost, SVM, LightGBM, Stacking, MLP, FT-Transformer, FT-Transformer-noAttn
6. Best (repeated-CV AUC): LR = 0.978 +- 0.006
7. LR holdout AUC = 0.983 (95% CI 0.973-0.991)
8. DeLong (Holm): 2/45 model pairs differ at alpha=0.05
9. Figures complete: True (7/7) | SHAP primary: LR
10. Files written to /content/drive/MyDrive/SoftCom_Myopia (69 files) incl. results.json, RESULTS_INDEX.md
Hybrid cohort (N=1500) - pipeline validation only, not clinical evidence
